# 19 · The agent loop with Gemini [LLM]

**Goal:** by the end you can connect real tools to Gemini, build the agent loop yourself (reason → act → observe), give
it memory for follow-up questions, stream its answer, and build the same agent in a few lines with `create_agent`.

**This lesson calls Gemini**, so it needs `GOOGLE_CLOUD_API_KEY` in your `.env` (see lesson 00). It makes about
**9 short LLM calls** in total.

## The idea in plain English

In lesson 18 *we* played the LLM and wrote the tool calls by hand. Now a real **LLM** (large language model), Gemini,
writes them. The graph is the same: a `model` node, a `ToolNode`, `tools_condition`, and an edge from the tools back
to the model.

**`bind_tools`** tells the LLM which tools exist: their names, descriptions and arguments (lesson 18, Step 2). The
LLM then decides, on its own, when to call a tool and with which arguments. This back-and-forth is often called a
**ReAct** loop, short for **re**ason + **act**:

1. **Reason**: the model reads the conversation and decides what to do next.
2. **Act**: it asks for a tool call.
3. **Observe**: the tool's result comes back as a `ToolMessage`, and the model reasons again.

It repeats until the model answers without asking for a tool. A program that runs this loop is called an **agent**.

**Analogy:** a student with a calculator doing a word problem. They read the problem, punch in one step, look at the
display, decide the next step, and write the answer when they're done.

## Picture

The same loop as lesson 18. Only the inside of `model` is different: a real LLM instead of a script.

```mermaid
flowchart LR
    S([START]) --> model["model (Gemini)"]
    model -. "tool calls: act" .-> tools["tools (ToolNode)"]
    tools -- "results: observe" --> model
    model -. "no tool calls: final answer" .-> E([END])
```

## Step 0: Connect to Gemini

`get_llm()` from `llm_setup.py` (lesson 00) is the only shared helper in the tutorial. It loads the key from `.env`
and returns a Gemini chat model. If the key is missing it stops here with a clear message, before any other code
runs.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from llm_setup import get_llm

llm = get_llm()
print("model:", llm.model)

API key loaded: True


model: gemini-2.5-flash-lite


## Step 1: The tools, and `bind_tools`

The same four tools as lesson 18. `llm.bind_tools(tools)` returns a new model object that sends the tool descriptions
along with every request.

In [2]:
from langchain.tools import tool


@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

Two more tools, then `bind_tools`. `llm_with_tools` is the model we'll use from now on.

In [3]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


tools = [add, subtract, multiply, divide]
llm_with_tools = llm.bind_tools(tools)

**LLM call 1.** We ask one question and look at the reply. Gemini doesn't answer with text: it replies with a **tool
call**, just like the `AIMessage` we wrote by hand in lesson 18. Nothing is executed yet. Calling the tool is the
graph's job, in Step 2.

`.text` is the reply's text (empty here), and `.tool_calls` lists the requested calls.

In [4]:
reply = llm_with_tools.invoke("What is 12.5 times 8?")

print("type:      ", type(reply).__name__)
print("text:      ", repr(reply.text))
print("tool_calls:", reply.tool_calls)

type:       AIMessage
text:       ''
tool_calls: [{'name': 'multiply', 'args': {'a': 12.5, 'b': 8}, 'id': 'de1233a4-e135-4811-abc4-db3d10705e2f', 'type': 'tool_call'}]


## Step 2: Build the agent loop by hand

### The model node

`call_model` sends the conversation so far to the LLM and appends its reply. In front of the conversation it puts a
**system message**: instructions for the model that the user doesn't see. Ours tells it to use the tools for every
step of arithmetic, rather than doing sums in its head.

In [5]:
from langchain.messages import SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState

SYSTEM_PROMPT = (
    "You are a calculator assistant. Use the tools for every arithmetic step, "
    "one step at a time. When you have the final result, answer in one short sentence."
)


def call_model(state: MessagesState) -> dict:
    messages = [SystemMessage(SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}

### The wiring

Exactly lesson 18's loop. `handle_tool_errors=True` turns a tool error (like dividing by zero) into a message the
model can read, instead of a crash.

In [6]:
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

agent = builder.compile()
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Run a multi-step question

**LLM calls 2–5 (about 4).** The question needs three steps that depend on each other: 3 + 5, then × 12, then ÷ 4.
Read the printout top to bottom and you'll see the loop: the model **reasons** and asks for one tool (**act**), the
`ToolMessage` brings back the result (**observe**), and so on, until a final answer with no tool calls.

(The exact wording of Gemini's replies can differ from run to run. The steps and the numbers shouldn't.)

In [7]:
from langchain.messages import HumanMessage

result = agent.invoke({"messages": [HumanMessage("What is (3 + 5) times 12, then divided by 4?")]})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is (3 + 5) times 12, then divided by 4?
================================== Ai Message ==================================
Tool Calls:
  add (c8781cba-e8d8-4708-a323-4800f3a7a784)
 Call ID: c8781cba-e8d8-4708-a323-4800f3a7a784
  Args:
    b: 5
    a: 3
================================= Tool Message =================================
Name: add

8.0
================================== Ai Message ==================================
Tool Calls:
  multiply (c5c1a7c2-1ba5-4ff2-b90e-e366675d6d4d)
 Call ID: c5c1a7c2-1ba5-4ff2-b90e-e366675d6d4d
  Args:
    b: 12
    a: 8
================================= Tool Message =================================
Name: multiply

96.0
================================== Ai Message ==================================
Tool Calls:
  divide (80959bcd-a4cd-478d-a6aa-07af109ba8c9)
 Call ID: 80959bcd-a4cd-478d-a6aa-07af109ba8c9
  Args:
    a: 96
    b: 4
================================

## Step 3: Memory for follow-up questions

A follow-up like "now add 10 to that" only makes sense if the agent remembers the conversation. That's the
checkpointer and `thread_id` from lesson 13. We compile the **same builder** again, this time with `InMemorySaver`.

To avoid paying to ask the first question again, we load the conversation from Step 2 into a new thread with
`update_state` (lesson 13). `as_node="model"` records it as if the `model` node had written it, so the thread is
finished and waiting for the next question.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "calc-chat-1"}}

memory_agent.update_state(config, {"messages": result["messages"]}, as_node="model")
snapshot = memory_agent.get_state(config)
print("messages in the thread:", len(snapshot.values["messages"]))
print("last message:          ", snapshot.values["messages"][-1].text)

messages in the thread: 8
last message:           The final answer is 24.0.


## Step 4: Stream the follow-up, token by token

**LLM calls 6–7.** We send the follow-up on the same thread, with `stream_mode="messages"` (lesson 15). Instead of
whole messages, this mode yields the LLM's reply in small pieces as they're generated: `(chunk, metadata)` pairs.
`metadata["langgraph_node"]` tells us which node produced the piece. We print the text of pieces from `model` as they
arrive, and note each tool call. Each `|` below separates two pieces.

In [9]:
follow_up = {"messages": [HumanMessage("Now add 10 to that.")]}

for chunk, metadata in memory_agent.stream(follow_up, config, stream_mode="messages"):
    if metadata["langgraph_node"] != "model":
        continue
    for call in getattr(chunk, "tool_calls", []):
        print(f"[tool call: {call['name']}({call['args']})]")
    if chunk.text:
        print(chunk.text, end="|")
print()

[tool call: add({'a': 24, 'b': 10})]


The final answer is 3|4.0.|


It remembered "that" = 24 from the earlier turns. Here's the end of the thread as the checkpointer saved it:

In [10]:
for message in memory_agent.get_state(config).values["messages"][-4:]:
    message.pretty_print()

================================ Human Message =================================

Now add 10 to that.
================================== Ai Message ==================================
Tool Calls:
  add (b4bf30ce-cbf8-4717-b4f6-0ada2444c59c)
 Call ID: b4bf30ce-cbf8-4717-b4f6-0ada2444c59c
  Args:
    a: 24
    b: 10
================================= Tool Message =================================
Name: add

34.0
================================== Ai Message ==================================

The final answer is 34.0.


## Step 5: The same agent in a few lines, `create_agent`

Building the loop by hand is the best way to understand it. For real programs, LangChain provides
**`create_agent`** (from `langchain.agents`). It builds the same graph for you (a model node, a tool node, the
routing and the loop) and returns a compiled graph you use like any other.

**LLM calls 8–9.** One question on a thread.

In [11]:
from langchain.agents import create_agent

quick_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)
print(quick_agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Using the agent is the same as using any compiled graph: `invoke` with messages and a thread config.

In [12]:
answer = quick_agent.invoke(
    {"messages": [HumanMessage("What is 2.5 times 4?")]},
    {"configurable": {"thread_id": "quick-1"}},
)
for message in answer["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 2.5 times 4?
================================== Ai Message ==================================
Tool Calls:
  multiply (fd35a5ad-9fa4-45e9-a22f-99be939336ce)
 Call ID: fd35a5ad-9fa4-45e9-a22f-99be939336ce
  Args:
    b: 4
    a: 2.5
================================= Tool Message =================================
Name: multiply

10.0
================================== Ai Message ==================================

The answer is 10.


Its diagram has the same shape as ours: `model`, `tools`, and a loop between them. You'll see
`create_react_agent` from `langgraph.prebuilt` in older tutorials. It's deprecated in LangGraph 1.x;
`create_agent` replaces it.

### How many LLM calls did we make?

Every `AIMessage` in a conversation is one LLM call. Counting them across the lesson:

In [13]:
from langchain.messages import AIMessage


def count_llm_calls(messages) -> int:
    return sum(isinstance(m, AIMessage) for m in messages)


follow_up_calls = count_llm_calls(memory_agent.get_state(config).values["messages"]) - count_llm_calls(result["messages"])
print("Step 1:", 1)
print("Step 2:", count_llm_calls(result["messages"]))
print("Step 4:", follow_up_calls)
print("Step 5:", count_llm_calls(answer["messages"]))
print("total: ", 1 + count_llm_calls(result["messages"]) + follow_up_calls + count_llm_calls(answer["messages"]))

Step 1: 1
Step 2: 4
Step 4: 2
Step 5: 2
total:  9


## What just happened

- **Step 1** bound four tools to Gemini. For "What is 12.5 times 8?" it replied with empty text and a tool call,
  `multiply` with `a=12.5, b=8`. Nothing ran yet.
- **Step 2** built the loop by hand. For the three-step question, the printout showed reason → act → observe three
  times (`add`, `multiply`, `divide`), then a final answer of 24.
- **Step 3** compiled the same builder with `InMemorySaver` and loaded the Step 2 conversation into thread
  `calc-chat-1` with `update_state`.
- **Step 4** streamed the follow-up "Now add 10 to that." with `stream_mode="messages"`: a tool call to `add` with
  24 and 10, then the answer 34, arriving in pieces.
- **Step 5** built the same agent with `create_agent` in one call. Its diagram has the same `model` ⇄ `tools` loop,
  and it answered "What is 2.5 times 4?" with the `multiply` tool.
- The count at the end shows how many LLM calls the lesson made.

## Common mistakes

1. **No API key.** `get_llm()` stops at Step 0 with:
   `AssertionError: GOOGLE_CLOUD_API_KEY is not set. Create a .env file in this folder with: ...`
   Fix: add the key to `.env` in the project folder (lesson 00), then restart the kernel.
2. **Using `llm` instead of `llm_with_tools` in the model node.** There's **no error**, but the model never sees the
   tools. It answers from its own head (sometimes wrongly) and the loop never visits `tools`.
   Fix: call `llm.bind_tools(tools)` and use the result in `call_model`.
3. **Copying `create_react_agent` from an older tutorial.** It's deprecated in LangGraph 1.x.
   Fix: `from langchain.agents import create_agent`, with `system_prompt=` for the instructions.

## Try it

1. Ask the hand-built `agent` "What is 10 divided by 0?" Does Gemini call the `divide` tool, or answer straight away?
   If it calls the tool, which message shows the tool's error? (This is one or two LLM calls.)
2. Continue the `quick_agent` thread `quick-1` with "Now subtract 3." Does it remember the 10? Print the last
   message. (About two LLM calls.)

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **LLM (large language model)**: a model that reads and writes text, such as Gemini.
- **`bind_tools`**: attaches tool descriptions to an LLM so it can ask for tool calls.
- **Agent**: a program in which an LLM decides, in a loop, which tools to call until it can answer.
- **ReAct**: "reason + act", the loop of reasoning, calling a tool and observing the result.
- **System message**: instructions for the model, placed before the conversation, that the user doesn't see.
- **`create_agent`**: LangChain's function that builds the agent loop graph for you, from a model and tools.
- **`stream_mode="messages"`**: streams an LLM's reply in small pieces (tokens) as `(chunk, metadata)` pairs.